## Установка GraphRAG + Ollama Host

In [1]:
!apt-get update -qq
!apt-get install -y zstd -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip install graphrag pyyaml onnxruntime -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.5/129.5 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.0/299.0 kB 6.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%#######                                                36.7%#################                              62.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Запуск Ollama Host

In [184]:
import subprocess
import time
import os

In [268]:
subprocess.run(["pkill", "-9", "ollama"], capture_output=True)

CompletedProcess(args=['pkill', '-9', 'ollama'], returncode=0, stdout=b'', stderr=b'')

In [269]:
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

log_file = open("/tmp/ollama.log", "w")
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file,
    start_new_session=True
)

In [270]:
!curl -s http://127.0.0.1:11434

Ollama is running

## Загрузка LLM

In [271]:
#!ollama pull llama3.1:8b

In [272]:
!ollama pull qwen2.5:7b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [273]:
#!ollama pull --insecure hf.co/bond005/meno-lite-0.1-gguf:Q4_K_M

## Загрузка модели эмбеддингов

In [274]:
#!ollama pull nomic-embed-text

In [275]:
!ollama pull bge-m3

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling daec91ffb5dd: 100% ▕██████████████████▏ 1.2 GB                         
pulling a406579cd136: 100% ▕██████████████████▏ 1.1 KB                         
pulling 0c4c9c2a325f: 100% ▕██████████████████▏  337 B                         
verifying sha256 digest 
writing manifest 
success 


## Подготовка структуры

In [276]:
import os
import shutil
import yaml

In [277]:
SRC_DIR = "/kaggle/input/datasets/eugeneklochko/md-from-mineru" 
PROJECT_DIR = "/kaggle/working/graph_rag"

In [278]:
os.makedirs(f"{PROJECT_DIR}/input", exist_ok=True)

In [297]:
settings = {
    "language": "Russian",
    "completion_models": {
        "default_completion_model": {
            "type": "litellm",
            "model_provider": "openai",
            "model": "qwen2.5:7b",
            "api_base": "http://localhost:11434/v1",
            "api_key": "not-needed",
            "max_tokens": 4096,
            "temperature": 0.1,
            "concurrent_requests": 1,
            "request_timeout": 600,
            "model_extra": {
                "drop_params": True,
                "num_ctx": 8192
            }
        }
    },
    "embedding_models": {
        "default_embedding_model": {
            "type": "litellm",
            "model_provider": "openai",
            "model": "bge-m3",
            "api_base": "http://localhost:11434/v1",
            "api_key": "not-needed",
            "batch_size": 8,
            "concurrent_requests": 1,
            "request_timeout": 120,
            "model_extra": {
                "drop_params": True,
                "num_ctx": 8192
            }
        }
    },
    "input": {
        "type": "text",
        "base_dir": "input",
        "file_pattern": ".*\\.md",
        "file_encoding": "utf-8"
    },
    "chunking": {
        "size": 600,
        "overlap": 120
    },
    "input_storage": {"type": "file", "base_dir": "input"},
    "output_storage": {"type": "file", "base_dir": "output/artifacts"},
    "cache": {"type": "json", "base_dir": "cache"},
    "reporting": {"type": "file", "base_dir": "output/reports"},
    
    "extract_graph": {
        "completion_model_id": "default_completion_model",
        "entity_types": [
            "ХИМИЧЕСКИЙ_ЭЛЕМЕНТ", 
            "МАТЕРИАЛ", 
            "МИКРОСТРУКТУРА", 
            "ТЕХНОЛОГИЧЕСКИЙ_ПРОЦЕСС", 
            "СВОЙСТВО", 
            "ОРГАНИЗАЦИЯ",
            "ПЕРСОНА"
        ],
        "max_gleanings": 0
    },
    
    "summarize_descriptions": {
        "completion_model_id": "default_completion_model",
        "max_length": 500,
        "max_gleaning_iterations": 1,
    },
    
    "community_reports": {
        "model_id": "default_completion_model",
        "max_length": 2000,
        "max_input_length": 8000
    },
    "embed_text": {
        "embedding_model_id": "default_embedding_model"
    },
    "local_search": {"max_tokens": 12000},
    "global_search": {
        "max_tokens": 12000,
        "data_max_tokens": 12000,
        "map_max_length": 1000,
        "reduce_max_length": 2000,
        "concurrency": 1
    },
}

In [298]:
with open(f"{PROJECT_DIR}/settings.yaml", "w", encoding="utf-8") as f:
    yaml.dump(settings, f, default_flow_style=False, allow_unicode=True)

In [281]:
shutil.copy("/kaggle/input/datasets/eugeneklochko/md-from-mineru/vanadiy_review.md", f"{PROJECT_DIR}/input/")

'/kaggle/working/graph_rag/input/vanadiy_review.md'

## Построение графа

In [299]:
for path in [
    f"{PROJECT_DIR}/cache",
    f"{PROJECT_DIR}/output",
    f"{PROJECT_DIR}/lancedb",
]:
    shutil.rmtree(path, ignore_errors=True)

Удалено: /kaggle/working/graph_rag/cache
Удалено: /kaggle/working/graph_rag/output
Удалено: /kaggle/working/graph_rag/prompts
Удалено: /kaggle/working/graph_rag/lancedb


In [300]:
log = open("/tmp/graphrag.log", "w")
proc = subprocess.Popen(
    ["graphrag", "index", "--root", f"{PROJECT_DIR}"],
    stdout=log,
    stderr=log,
    start_new_session=True
)
print("GraphRAG запущен, PID:", proc.pid)

GraphRAG запущен, PID: 5653


In [305]:
!tail -f /tmp/graphrag.log


Workflow complete: extract_covariates
Starting workflow: create_communities

Workflow complete: create_communities
Starting workflow: create_final_text_units

Workflow complete: create_final_text_units
Starting workflow: create_community_reports
[2026-09-21T15:53:31Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/graph_rag/output/lancedb/entity_description.lance, it will be created

Workflow complete: create_community_reports
Starting workflow: generate_text_embeddings
[2026-09-21T15:53:40Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/graph_rag/output/lancedb/community_full_content.lance, it will be created
[2026-09-21T15:53:45Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/graph_rag/output/lancedb/text_unit_text.lance, it will be created
  2 / 2 ............................................................................................
Workflow complete: generate_text_embeddings
Pipeline comple

In [237]:
!pkill -9 -f graphrag

In [306]:
!rm /kaggle/working/output.zip

In [307]:
!cd {PROJECT_DIR} && zip -r /kaggle/working/output.zip output

  adding: output/ (stored 0%)
  adding: output/artifacts/ (stored 0%)
  adding: output/artifacts/entities.parquet (deflated 28%)
  adding: output/artifacts/context.json (stored 0%)
  adding: output/artifacts/communities.parquet (deflated 55%)
  adding: output/artifacts/community_reports.parquet (deflated 39%)
  adding: output/artifacts/relationships.parquet (deflated 36%)
  adding: output/artifacts/documents.parquet (deflated 20%)
  adding: output/artifacts/text_units.parquet (deflated 33%)
  adding: output/artifacts/stats.json (deflated 76%)
  adding: output/reports/ (stored 0%)
  adding: output/reports/indexing-engine.log (deflated 88%)
  adding: output/lancedb/ (stored 0%)
  adding: output/lancedb/entity_description.lance/ (stored 0%)
  adding: output/lancedb/entity_description.lance/_versions/ (stored 0%)
  adding: output/lancedb/entity_description.lance/_versions/18446744073709551614.manifest (deflated 75%)
  adding: output/lancedb/entity_description.lance/_versions/18446744073709